# Video Game Commercial Success Prediction & Market Analysis

The goal of this project is to deliver a clear commercial narrative that game studios and publishers can use to minimize financial risk before greenlighting a new game's production.

When looking into creating new games studios and publishers investigate what has been historically successful. By analyzing characteristics of games that have already been released, we can help guide new game ideas to the right publishers to help propagate the game to a better sales pattern, allowing both the publisher and game developer to maximize their investment.

#### Imports

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

## Step 1: Environment Setup & In-Memory Database Initialization (SQL)

To start, we will create an in-memory database using SQLAlchemy. 
By doing this, the data pipeline can be run anywhere.

In [ ]:
# Create an in memory SQLite database
engine = create_engine('sqlite:///:memory:')

In [ ]:
# Load data into pandas dataframe
df_raw = pd.read_csv('datasets/Video_Games_Sales.csv')
df_raw.info()

In [ ]:
# ingest raw data into the in-memory SQL database
df_raw.to_sql('raw_game_sales', con=engine, index=False, if_exists='replace')

In [ ]:
# Run SQL query directly against the database
query = """
SELECT count(*)
FROM raw_game_sales
"""

In [ ]:
with engine.connect() as conn:
    results = conn.execute(text(query))
    print(f'Successfully ingested {results.scalar()} records into in-memory table "raw_game_sales".')

## Step 2: Cleaning and Transformation (SQL)

Next we will pull the columns we need for the analysis. We will also filter out the `NULL` values from the `critic_scores`, `global_sales`, `year_of_release`, `user_score`, and `publisher`.

In [ ]:
# Query to pull only the needed columns from database
query = """
SELECT name as name,
    platform as platform,
    year_of_release as release_year,
    genre as genre, 
    publisher as publisher,
    na_sales as na_sales,
    eu_sales as eu_sales,
    jp_sales as jp_sales,
    other_sales as other_sales,
    global_sales as global_sales,
    critic_score as critic_score,
    user_score as user_score
FROM raw_game_sales
WHERE critic_score IS NOT NULL 
AND global_sales IS NOT NULL
AND year_of_release IS NOT NULL
AND user_score != 'tbd'
AND publisher IS NOT NULL;
"""

In [ ]:
with engine.connect() as conn:
    results = conn.execute(text(query))
    df_cleaned = pd.DataFrame(results)

In [ ]:
df_cleaned.info()

Looking above, we see that the `user_score` is of the type `object (string)`. We need to convert this to a numeric type to align the data.

In [ ]:
# Change column to numeric
df_cleaned['user_score'] = pd.to_numeric(df_cleaned['user_score'], errors='coerce')

In [ ]:
# Remove rows containing NaN values
df_cleaned = df_cleaned.dropna()

In [ ]:
# Confirm change
df_cleaned.info()

## Step 3: Exploratory Data Analysis & Feature Engineering (Python)

Now we need to set up the binary classification target: `Is_Hit` = `1` if `global_sales` $\geq 1.0$ else `0`.

In [ ]:
# create 'is_hit' column setting games that gloably made more than 1 million dollers to 1 and 0 if not
df_cleaned['is_hit'] = np.where(df_cleaned['global_sales'] >= 1.0, 1, 0)

In [ ]:
# confirm column creation and value placement
df_cleaned.sample(10)

Next we'll calculate the baseline class balance ratio (percetage of hits vs. non-hits).

In [ ]:
# Calulate the average and multiply by 100 to get the percentage
counts = df_cleaned['is_hit'].value_counts(normalize=True) * 100

print('Balance Ratios:')
print(f'hit:      {counts.get(1.0) :.2f}%')
print(f'not_hit: {counts.get(0.0) : .2f}%')

Above, we calculate the class balance ratios using `normalize=True`, then multiply by 100 to yield relative percentages rather than raw counts. By referencing values with explicit class labels (`1` for hit, `0` for non-hit) rather than positional indexing, the calculation remains accurate and robust to unexpected shifts in class distribution in future data ingestions.

#### Feature Selection & Preprocessing

In [ ]:
df_cleaned.columns

In [ ]:
# target vector
y = df_cleaned['is_hit']

# predictive features
features = ['platform','release_year','genre','publisher','critic_score','user_score']

X = df_cleaned[features]

> Note: The regional sales columns (`na_sales`, `eu_sales`, `jp_sales`, `other_sales`) and `global_sales` are retained in the cleaned SQL dataset for exploratory data analysis and regional marketing profiling. To prevent target leakage, all sales metrics will be strictly excluded from the feature matrix (`X`) prior to model training.